# ANSI SQL Using MySQL — Module 1
<!-- ## 25 Exercises with Query Output

**No MySQL needed** — uses a local SQLite database file (`event_management.db`)  
**Tables:** `Users` · `Events` · `Sessions` · `Registrations` · `Feedback` · `Resources`

> **How to run:** open in VS Code → select Python kernel → click **Run All** -->

In [1]:
import sys
!{sys.executable} -m pip install pandas --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup — Database, Tables & Sample Data
Creates `event_management.db` in the same folder. Safe to re-run.

In [2]:
import sqlite3, os, pandas as pd
from IPython.display import display

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "event_management.db")
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")
print(f"Database: {DB_PATH}")

ddl = """
CREATE TABLE IF NOT EXISTS Users (
    user_id           INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name         TEXT NOT NULL,
    email             TEXT NOT NULL UNIQUE,
    city              TEXT NOT NULL,
    registration_date TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Events (
    event_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    title        TEXT NOT NULL,
    description  TEXT,
    city         TEXT NOT NULL,
    start_date   TEXT NOT NULL,
    end_date     TEXT NOT NULL,
    status       TEXT CHECK(status IN ('upcoming','completed','cancelled')),
    organizer_id INTEGER REFERENCES Users(user_id)
);
CREATE TABLE IF NOT EXISTS Sessions (
    session_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    event_id     INTEGER REFERENCES Events(event_id),
    title        TEXT NOT NULL,
    speaker_name TEXT NOT NULL,
    start_time   TEXT NOT NULL,
    end_time     TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Registrations (
    registration_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id           INTEGER REFERENCES Users(user_id),
    event_id          INTEGER REFERENCES Events(event_id),
    registration_date TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Feedback (
    feedback_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id       INTEGER REFERENCES Users(user_id),
    event_id      INTEGER REFERENCES Events(event_id),
    rating        INTEGER CHECK(rating BETWEEN 1 AND 5),
    comments      TEXT,
    feedback_date TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Resources (
    resource_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    event_id      INTEGER REFERENCES Events(event_id),
    resource_type TEXT CHECK(resource_type IN ('pdf','image','link')),
    resource_url  TEXT NOT NULL,
    uploaded_at   TEXT NOT NULL
);
"""
for stmt in ddl.strip().split(";"):
    stmt = stmt.strip()
    if stmt:
        conn.execute(stmt)

inserts = [
    "INSERT OR IGNORE INTO Users VALUES (1,'Alice Johnson','alice@example.com','New York','2024-12-01')",
    "INSERT OR IGNORE INTO Users VALUES (2,'Bob Smith','bob@example.com','Los Angeles','2024-12-05')",
    "INSERT OR IGNORE INTO Users VALUES (3,'Charlie Lee','charlie@example.com','Chicago','2024-12-10')",
    "INSERT OR IGNORE INTO Users VALUES (4,'Diana King','diana@example.com','New York','2025-01-15')",
    "INSERT OR IGNORE INTO Users VALUES (5,'Ethan Hunt','ethan@example.com','Los Angeles','2025-02-01')",
    "INSERT OR IGNORE INTO Events VALUES (1,'Tech Innovators Meetup','A meetup for tech enthusiasts.','New York','2025-06-10 10:00:00','2025-06-10 16:00:00','upcoming',1)",
    "INSERT OR IGNORE INTO Events VALUES (2,'AI & ML Conference','Conference on AI and ML advancements.','Chicago','2025-05-15 09:00:00','2025-05-15 17:00:00','completed',3)",
    "INSERT OR IGNORE INTO Events VALUES (3,'Frontend Development Bootcamp','Hands-on training on frontend tech.','Los Angeles','2025-07-01 10:00:00','2025-07-03 16:00:00','upcoming',2)",
    "INSERT OR IGNORE INTO Sessions VALUES (1,1,'Opening Keynote','Dr. Tech','2025-06-10 10:00:00','2025-06-10 11:00:00')",
    "INSERT OR IGNORE INTO Sessions VALUES (2,1,'Future of Web Dev','Alice Johnson','2025-06-10 11:15:00','2025-06-10 12:30:00')",
    "INSERT OR IGNORE INTO Sessions VALUES (3,2,'AI in Healthcare','Charlie Lee','2025-05-15 09:30:00','2025-05-15 11:00:00')",
    "INSERT OR IGNORE INTO Sessions VALUES (4,3,'Intro to HTML5','Bob Smith','2025-07-01 10:00:00','2025-07-01 12:00:00')",
    "INSERT OR IGNORE INTO Registrations VALUES (1,1,1,'2025-05-01')",
    "INSERT OR IGNORE INTO Registrations VALUES (2,2,1,'2025-05-02')",
    "INSERT OR IGNORE INTO Registrations VALUES (3,3,2,'2025-04-30')",
    "INSERT OR IGNORE INTO Registrations VALUES (4,4,2,'2025-04-28')",
    "INSERT OR IGNORE INTO Registrations VALUES (5,5,3,'2025-06-15')",
    "INSERT OR IGNORE INTO Feedback VALUES (1,3,2,4,'Great insights!','2025-05-16')",
    "INSERT OR IGNORE INTO Feedback VALUES (2,4,2,5,'Very informative.','2025-05-16')",
    "INSERT OR IGNORE INTO Feedback VALUES (3,2,1,3,'Could be better.','2025-06-11')",
    "INSERT OR IGNORE INTO Resources VALUES (1,1,'pdf','https://portal.com/resources/tech_meetup_agenda.pdf','2025-05-01 10:00:00')",
    "INSERT OR IGNORE INTO Resources VALUES (2,2,'image','https://portal.com/resources/ai_poster.jpg','2025-04-20 09:00:00')",
    "INSERT OR IGNORE INTO Resources VALUES (3,3,'link','https://portal.com/resources/html5_docs','2025-06-25 15:00:00')",
]
for ins in inserts:
    conn.execute(ins)
conn.commit()
print("Schema + sample data ready.")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


Database: d:\College\placements\Cognizant\Digital-Nurture-Python\Solutions\SQL\event_management.db
Schema + sample data ready.


## Helper

In [3]:
def run_query(sql):
    df = pd.read_sql_query(sql, conn)
    if df.empty:
        print("  (no rows returned)")
    else:
        display(df)
    return df


---
## Exercises

### Exercise 1: User Upcoming Events
All upcoming events a user is registered for **in their city**, sorted by start date.

In [4]:
sql = """
SELECT u.user_id, u.full_name,
       e.title AS event_title, e.city, e.start_date
FROM   Users u
JOIN   Registrations r ON r.user_id  = u.user_id
JOIN   Events        e ON e.event_id = r.event_id
WHERE  e.status = 'upcoming'
  AND  e.city   = u.city
ORDER  BY e.start_date
"""
print(sql)
run_query(sql)


SELECT u.user_id, u.full_name,
       e.title AS event_title, e.city, e.start_date
FROM   Users u
JOIN   Registrations r ON r.user_id  = u.user_id
JOIN   Events        e ON e.event_id = r.event_id
WHERE  e.status = 'upcoming'
  AND  e.city   = u.city
ORDER  BY e.start_date



,user_id,full_name,event_title,city,start_date
0,1,Alice Johnson,Tech Innovators Meetup,New York,2025-06-10 10:00:00
1,5,Ethan Hunt,Frontend Development Bootcamp,Los Angeles,2025-07-01 10:00:00


,user_id,full_name,event_title,city,start_date
0,1,Alice Johnson,Tech Innovators Meetup,New York,2025-06-10 10:00:00
1,5,Ethan Hunt,Frontend Development Bootcamp,Los Angeles,2025-07-01 10:00:00


### Exercise 2: Top Rated Events
Events with the highest average rating having **at least 10** feedback submissions.

In [5]:
sql = """
SELECT e.event_id, e.title,
       ROUND(AVG(f.rating), 2) AS avg_rating,
       COUNT(f.feedback_id)    AS feedback_count
FROM   Events   e
JOIN   Feedback f ON f.event_id = e.event_id
GROUP  BY e.event_id, e.title
HAVING COUNT(f.feedback_id) >= 10
ORDER  BY avg_rating DESC
"""
print(sql)
run_query(sql)


SELECT e.event_id, e.title,
       ROUND(AVG(f.rating), 2) AS avg_rating,
       COUNT(f.feedback_id)    AS feedback_count
FROM   Events   e
JOIN   Feedback f ON f.event_id = e.event_id
GROUP  BY e.event_id, e.title
HAVING COUNT(f.feedback_id) >= 10
ORDER  BY avg_rating DESC

  (no rows returned)


,event_id,title,avg_rating,feedback_count


### Exercise 3: Inactive Users
Users who have **not registered** for any event in the last 90 days.

In [6]:
sql = """
SELECT u.user_id, u.full_name, u.email
FROM   Users u
WHERE  u.user_id NOT IN (
    SELECT r.user_id
    FROM   Registrations r
    WHERE  r.registration_date >= DATE('now', '-90 days')
)
"""
print(sql)
run_query(sql)


SELECT u.user_id, u.full_name, u.email
FROM   Users u
WHERE  u.user_id NOT IN (
    SELECT r.user_id
    FROM   Registrations r
    WHERE  r.registration_date >= DATE('now', '-90 days')
)



,user_id,full_name,email
0,1,Alice Johnson,alice@example.com
1,2,Bob Smith,bob@example.com
2,3,Charlie Lee,charlie@example.com
3,4,Diana King,diana@example.com
4,5,Ethan Hunt,ethan@example.com


,user_id,full_name,email
0,1,Alice Johnson,alice@example.com
1,2,Bob Smith,bob@example.com
2,3,Charlie Lee,charlie@example.com
3,4,Diana King,diana@example.com
4,5,Ethan Hunt,ethan@example.com


### Exercise 4: Peak Session Hours
Sessions scheduled between **10 AM and 12 PM** per event.

In [7]:
sql = """
SELECT e.event_id, e.title,
       COUNT(s.session_id) AS peak_hour_sessions
FROM   Events   e
JOIN   Sessions s ON s.event_id = e.event_id
WHERE  CAST(strftime('%H', s.start_time) AS INTEGER) >= 10
  AND  CAST(strftime('%H', s.start_time) AS INTEGER) <  12
GROUP  BY e.event_id, e.title
"""
print(sql)
run_query(sql)


SELECT e.event_id, e.title,
       COUNT(s.session_id) AS peak_hour_sessions
FROM   Events   e
JOIN   Sessions s ON s.event_id = e.event_id
WHERE  CAST(strftime('%H', s.start_time) AS INTEGER) >= 10
  AND  CAST(strftime('%H', s.start_time) AS INTEGER) <  12
GROUP  BY e.event_id, e.title



,event_id,title,peak_hour_sessions
0,1,Tech Innovators Meetup,2
1,3,Frontend Development Bootcamp,1


,event_id,title,peak_hour_sessions
0,1,Tech Innovators Meetup,2
1,3,Frontend Development Bootcamp,1


### Exercise 5: Most Active Cities
Top 5 cities by **distinct user registrations**.

In [8]:
sql = """
SELECT u.city,
       COUNT(DISTINCT r.user_id) AS distinct_registrations
FROM   Users         u
JOIN   Registrations r ON r.user_id = u.user_id
GROUP  BY u.city
ORDER  BY distinct_registrations DESC
LIMIT  5
"""
print(sql)
run_query(sql)


SELECT u.city,
       COUNT(DISTINCT r.user_id) AS distinct_registrations
FROM   Users         u
JOIN   Registrations r ON r.user_id = u.user_id
GROUP  BY u.city
ORDER  BY distinct_registrations DESC
LIMIT  5



,city,distinct_registrations
0,New York,2
1,Los Angeles,2
2,Chicago,1


,city,distinct_registrations
0,New York,2
1,Los Angeles,2
2,Chicago,1


### Exercise 6: Event Resource Summary
PDFs, images, and links uploaded per event.

In [9]:
sql = """
SELECT e.event_id, e.title,
       SUM(CASE WHEN r.resource_type='pdf'   THEN 1 ELSE 0 END) AS pdf_count,
       SUM(CASE WHEN r.resource_type='image' THEN 1 ELSE 0 END) AS image_count,
       SUM(CASE WHEN r.resource_type='link'  THEN 1 ELSE 0 END) AS link_count,
       COUNT(r.resource_id)                                      AS total_resources
FROM   Events    e
LEFT JOIN Resources r ON r.event_id = e.event_id
GROUP  BY e.event_id, e.title
"""
print(sql)
run_query(sql)


SELECT e.event_id, e.title,
       SUM(CASE WHEN r.resource_type='pdf'   THEN 1 ELSE 0 END) AS pdf_count,
       SUM(CASE WHEN r.resource_type='image' THEN 1 ELSE 0 END) AS image_count,
       SUM(CASE WHEN r.resource_type='link'  THEN 1 ELSE 0 END) AS link_count,
       COUNT(r.resource_id)                                      AS total_resources
FROM   Events    e
LEFT JOIN Resources r ON r.event_id = e.event_id
GROUP  BY e.event_id, e.title



,event_id,title,pdf_count,image_count,link_count,total_resources
0,1,Tech Innovators Meetup,1,0,0,1
1,2,AI & ML Conference,0,1,0,1
2,3,Frontend Development Bootcamp,0,0,1,1


,event_id,title,pdf_count,image_count,link_count,total_resources
0,1,Tech Innovators Meetup,1,0,0,1
1,2,AI & ML Conference,0,1,0,1
2,3,Frontend Development Bootcamp,0,0,1,1


### Exercise 7: Low Feedback Alerts
Users who gave a rating **< 3**, with comments and event name.

In [10]:
sql = """
SELECT u.user_id, u.full_name,
       e.title AS event_title,
       f.rating, f.comments, f.feedback_date
FROM   Feedback f
JOIN   Users  u ON u.user_id  = f.user_id
JOIN   Events e ON e.event_id = f.event_id
WHERE  f.rating < 3
"""
print(sql)
run_query(sql)


SELECT u.user_id, u.full_name,
       e.title AS event_title,
       f.rating, f.comments, f.feedback_date
FROM   Feedback f
JOIN   Users  u ON u.user_id  = f.user_id
JOIN   Events e ON e.event_id = f.event_id
WHERE  f.rating < 3

  (no rows returned)


,user_id,full_name,event_title,rating,comments,feedback_date


### Exercise 8: Sessions per Upcoming Event
Upcoming events with their session count.

In [11]:
sql = """
SELECT e.event_id, e.title,
       COUNT(s.session_id) AS session_count
FROM   Events e
LEFT JOIN Sessions s ON s.event_id = e.event_id
WHERE  e.status = 'upcoming'
GROUP  BY e.event_id, e.title
"""
print(sql)
run_query(sql)


SELECT e.event_id, e.title,
       COUNT(s.session_id) AS session_count
FROM   Events e
LEFT JOIN Sessions s ON s.event_id = e.event_id
WHERE  e.status = 'upcoming'
GROUP  BY e.event_id, e.title



,event_id,title,session_count
0,1,Tech Innovators Meetup,2
1,3,Frontend Development Bootcamp,1


,event_id,title,session_count
0,1,Tech Innovators Meetup,2
1,3,Frontend Development Bootcamp,1


### Exercise 9: Organizer Event Summary
For each organizer, count of events per status.

In [12]:
sql = """
SELECT u.user_id   AS organizer_id,
       u.full_name AS organizer_name,
       e.status,
       COUNT(e.event_id) AS event_count
FROM   Users  u
JOIN   Events e ON e.organizer_id = u.user_id
GROUP  BY u.user_id, u.full_name, e.status
ORDER  BY u.user_id, e.status
"""
print(sql)
run_query(sql)


SELECT u.user_id   AS organizer_id,
       u.full_name AS organizer_name,
       e.status,
       COUNT(e.event_id) AS event_count
FROM   Users  u
JOIN   Events e ON e.organizer_id = u.user_id
GROUP  BY u.user_id, u.full_name, e.status
ORDER  BY u.user_id, e.status



,organizer_id,organizer_name,status,event_count
0,1,Alice Johnson,upcoming,1
1,2,Bob Smith,upcoming,1
2,3,Charlie Lee,completed,1


,organizer_id,organizer_name,status,event_count
0,1,Alice Johnson,upcoming,1
1,2,Bob Smith,upcoming,1
2,3,Charlie Lee,completed,1


### Exercise 10: Feedback Gap
Events that had registrations but received **no feedback**.

In [13]:
sql = """
SELECT e.event_id, e.title,
       COUNT(DISTINCT r.registration_id) AS total_registrations
FROM   Events        e
JOIN   Registrations r  ON r.event_id = e.event_id
LEFT JOIN Feedback   f  ON f.event_id = e.event_id
WHERE  f.feedback_id IS NULL
GROUP  BY e.event_id, e.title
"""
print(sql)
run_query(sql)


SELECT e.event_id, e.title,
       COUNT(DISTINCT r.registration_id) AS total_registrations
FROM   Events        e
JOIN   Registrations r  ON r.event_id = e.event_id
LEFT JOIN Feedback   f  ON f.event_id = e.event_id
WHERE  f.feedback_id IS NULL
GROUP  BY e.event_id, e.title



,event_id,title,total_registrations
0,3,Frontend Development Bootcamp,1


,event_id,title,total_registrations
0,3,Frontend Development Bootcamp,1


### Exercise 11: Daily New User Count
New users registered **each day in the last 7 days**.

In [14]:
sql = """
SELECT registration_date,
       COUNT(user_id) AS new_users
FROM   Users
WHERE  registration_date >= DATE('now', '-7 days')
GROUP  BY registration_date
ORDER  BY registration_date
"""
print(sql)
run_query(sql)


SELECT registration_date,
       COUNT(user_id) AS new_users
FROM   Users
WHERE  registration_date >= DATE('now', '-7 days')
GROUP  BY registration_date
ORDER  BY registration_date

  (no rows returned)


,registration_date,new_users


### Exercise 12: Event with Maximum Sessions
Event(s) with the **highest** number of sessions.

In [15]:
sql = """
SELECT e.event_id, e.title,
       COUNT(s.session_id) AS session_count
FROM   Events   e
JOIN   Sessions s ON s.event_id = e.event_id
GROUP  BY e.event_id, e.title
HAVING COUNT(s.session_id) = (
    SELECT MAX(cnt) FROM (
        SELECT COUNT(session_id) AS cnt
        FROM   Sessions GROUP BY event_id
    )
)
"""
print(sql)
run_query(sql)


SELECT e.event_id, e.title,
       COUNT(s.session_id) AS session_count
FROM   Events   e
JOIN   Sessions s ON s.event_id = e.event_id
GROUP  BY e.event_id, e.title
HAVING COUNT(s.session_id) = (
    SELECT MAX(cnt) FROM (
        SELECT COUNT(session_id) AS cnt
        FROM   Sessions GROUP BY event_id
    )
)



,event_id,title,session_count
0,1,Tech Innovators Meetup,2


,event_id,title,session_count
0,1,Tech Innovators Meetup,2


### Exercise 13: Average Rating per City
Average feedback rating of events held in each city.

In [ ]:
sql = """
SELECT e.city,
       ROUND(AVG(f.rating), 2) AS avg_rating
FROM   Events   e
JOIN   Feedback f ON f.event_id = e.event_id
GROUP  BY e.city
ORDER  BY avg_rating DESC
"""
print(sql)
run_query(sql)

### Exercise 14: Most Registered Events
Top 3 events by total registrations.

In [ ]:
sql = """
SELECT e.event_id, e.title,
       COUNT(r.registration_id) AS total_registrations
FROM   Events        e
JOIN   Registrations r ON r.event_id = e.event_id
GROUP  BY e.event_id, e.title
ORDER  BY total_registrations DESC
LIMIT  3
"""
print(sql)
run_query(sql)

### Exercise 15: Event Session Time Conflict
Pairs of sessions in the **same event** whose times overlap.

In [ ]:
sql = """
SELECT s1.event_id,
       s1.session_id AS session_a, s1.title AS title_a,
       s1.start_time AS start_a,  s1.end_time AS end_a,
       s2.session_id AS session_b, s2.title AS title_b,
       s2.start_time AS start_b,  s2.end_time AS end_b
FROM   Sessions s1
JOIN   Sessions s2
  ON   s1.event_id   = s2.event_id
 AND   s1.session_id < s2.session_id
 AND   s1.start_time < s2.end_time
 AND   s1.end_time   > s2.start_time
"""
print(sql)
run_query(sql)

### Exercise 16: Unregistered Active Users
Users who signed up in the **last 30 days** but haven't registered for any event.

In [ ]:
sql = """
SELECT u.user_id, u.full_name, u.registration_date
FROM   Users u
WHERE  u.registration_date >= DATE('now', '-30 days')
  AND  u.user_id NOT IN (SELECT DISTINCT user_id FROM Registrations)
"""
print(sql)
run_query(sql)

### Exercise 17: Multi-Session Speakers
Speakers handling **more than one session** across all events.

In [ ]:
sql = """
SELECT speaker_name,
       COUNT(session_id) AS session_count
FROM   Sessions
GROUP  BY speaker_name
HAVING COUNT(session_id) > 1
ORDER  BY session_count DESC
"""
print(sql)
run_query(sql)

### Exercise 18: Resource Availability Check
Events that have **no resources** uploaded.

In [ ]:
sql = """
SELECT e.event_id, e.title
FROM   Events e
LEFT JOIN Resources r ON r.event_id = e.event_id
WHERE  r.resource_id IS NULL
"""
print(sql)
run_query(sql)

### Exercise 19: Completed Events with Feedback Summary
For completed events: total registrations and average rating.

In [ ]:
sql = """
SELECT e.event_id, e.title,
       COUNT(DISTINCT r.registration_id) AS total_registrations,
       ROUND(AVG(f.rating), 2)           AS avg_rating
FROM   Events e
LEFT JOIN Registrations r ON r.event_id = e.event_id
LEFT JOIN Feedback      f ON f.event_id = e.event_id
WHERE  e.status = 'completed'
GROUP  BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 20: User Engagement Index
For each user: events registered and feedbacks submitted.

In [ ]:
sql = """
SELECT u.user_id, u.full_name,
       COUNT(DISTINCT r.event_id)    AS events_registered,
       COUNT(DISTINCT f.feedback_id) AS feedbacks_submitted
FROM   Users u
LEFT JOIN Registrations r ON r.user_id = u.user_id
LEFT JOIN Feedback      f ON f.user_id = u.user_id
GROUP  BY u.user_id, u.full_name
ORDER  BY events_registered DESC, feedbacks_submitted DESC
"""
print(sql)
run_query(sql)

### Exercise 21: Top Feedback Providers
Top 5 users by feedback entries submitted.

In [ ]:
sql = """
SELECT u.user_id, u.full_name,
       COUNT(f.feedback_id) AS feedback_count
FROM   Users    u
JOIN   Feedback f ON f.user_id = u.user_id
GROUP  BY u.user_id, u.full_name
ORDER  BY feedback_count DESC
LIMIT  5
"""
print(sql)
run_query(sql)

### Exercise 22: Duplicate Registrations Check
Users registered **more than once** for the same event.

In [ ]:
sql = """
SELECT user_id, event_id,
       COUNT(*) AS registration_count
FROM   Registrations
GROUP  BY user_id, event_id
HAVING COUNT(*) > 1
"""
print(sql)
run_query(sql)

### Exercise 23: Registration Trends
Month-wise registration count over the **past 12 months**.

In [ ]:
sql = """
SELECT strftime('%Y-%m', registration_date) AS month,
       COUNT(registration_id)               AS registrations
FROM   Registrations
WHERE  registration_date >= DATE('now', '-12 months')
GROUP  BY strftime('%Y-%m', registration_date)
ORDER  BY month
"""
print(sql)
run_query(sql)

### Exercise 24: Average Session Duration per Event
Average session duration **(in minutes)** per event.

In [ ]:
sql = """
SELECT e.event_id, e.title,
       ROUND(AVG(
           (julianday(s.end_time) - julianday(s.start_time)) * 24 * 60
       ), 2) AS avg_duration_minutes
FROM   Events   e
JOIN   Sessions s ON s.event_id = e.event_id
GROUP  BY e.event_id, e.title
"""
print(sql)
run_query(sql)

### Exercise 25: Events Without Sessions
Events that have **no sessions** scheduled.

In [ ]:
sql = """
SELECT e.event_id, e.title, e.status
FROM   Events e
LEFT JOIN Sessions s ON s.event_id = e.event_id
WHERE  s.session_id IS NULL
"""
print(sql)
run_query(sql)